# 04 - Análise de Dados (Camada Gold)

## Objetivo
Responder às perguntas de negócio definidas na etapa de objetivo do MVP,
utilizando as tabelas do esquema estrela da camada Gold. A análise
cobre também a verificação de qualidade final dos dados modelados.

## Perguntas de negócio
1. Qual o perfil das internações por sexo?
2. Qual o perfil das internações por faixa etária?
3. Quais os principais diagnósticos (CID-10) das internações?
4. Existe sazonalidade nas internações ao longo do ano?
5. Qual a permanência média e como ela varia por diagnóstico?
6. Quais os custos totais e médios das internações?
Complemento: taxa de mortalidade e uso de UTI.

## Fonte dos dados
Tabelas da camada Gold, registradas no catálogo:
fato_internacoes, dim_tempo, dim_diagnostico, dim_municipio, dim_hospital.

In [0]:
# Configuração da análise.
# O catálogo e o schema são detectados do ambiente para montar os nomes
# completos das tabelas, sem valores fixos.
from pyspark.sql.functions import col, count

catalogo = spark.catalog.currentCatalog()
schema = spark.catalog.currentDatabase()
base = f"{catalogo}.{schema}"
print("Catálogo:", catalogo, "| Schema:", schema)

# Contagem de cada tabela da Gold, confirmando a leitura do modelo.
for tabela in ["dim_tempo", "dim_diagnostico", "dim_municipio", "dim_hospital", "fato_internacoes"]:
    total = spark.table(f"{base}.{tabela}").count()
    print(f"{tabela}: {total:,} linhas".replace(",", "."))

In [0]:
# Verificação de qualidade na camada Gold.
# São checados valores negativos em campos que não aceitam valores
# negativos, permanências negativas (inconsistência de datas) e
# categorias de sexo e idade não informadas.
print("Checagens de qualidade na Gold:")
spark.sql(f"""
SELECT
  COUNT(*)                                        AS total_internacoes,
  SUM(CASE WHEN VAL_TOT < 0        THEN 1 ELSE 0 END) AS valor_total_negativo,
  SUM(CASE WHEN permanencia_dias < 0 THEN 1 ELSE 0 END) AS permanencia_negativa,
  SUM(CASE WHEN idade_anos < 0     THEN 1 ELSE 0 END) AS idade_negativa,
  SUM(CASE WHEN sexo_desc = 'Ignorado'   THEN 1 ELSE 0 END) AS sexo_ignorado,
  SUM(CASE WHEN faixa_etaria = 'Ignorado' THEN 1 ELSE 0 END) AS idade_ignorada
FROM {base}.fato_internacoes
""").show(truncate=False)

### Pergunta 1: Qual o perfil das internações por sexo?

In [0]:
# Percentual de internações por sexo.
# A janela SUM(COUNT(*)) OVER() calcula o total geral dentro da própria
# consulta, permitindo o percentual sem uma segunda leitura da tabela.
spark.sql(f"""
SELECT sexo_desc AS sexo,
       COUNT(*)   AS internacoes,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM {base}.fato_internacoes
GROUP BY sexo_desc
ORDER BY internacoes DESC
""").show(truncate=False)

### Pergunta 2: Qual o perfil das internações por faixa etária?

In [0]:
# Percentual de internações por faixa etária.
# A coluna idade_min é calculada no SELECT para servir de critério de
# ordenação, mantendo a ordem natural das faixas (0-4, 5-9, 10-19...).
# A ordenação direta por MIN(idade_anos) falha quando a consulta possui
# função de janela, pois o ORDER BY é resolvido sobre a projeção final.
# A categoria Ignorado, sem idade, fica por último devido ao NULLS LAST.
spark.sql(f"""
SELECT faixa_etaria,
       COUNT(*) AS internacoes,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct,
       MIN(idade_anos) AS idade_min
FROM {base}.fato_internacoes
GROUP BY faixa_etaria
ORDER BY idade_min NULLS LAST
""").show(truncate=False)

### Pergunta 3: Quais os principais diagnósticos (CID-10) das internações?

In [0]:
# Top 10 capítulos CID-10 por volume de internações.
spark.sql(f"""
SELECT d.capitulo_cid AS capitulo,
       COUNT(*) AS internacoes,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM {base}.fato_internacoes f
JOIN {base}.dim_diagnostico d ON f.sk_diagnostico = d.sk_diagnostico
GROUP BY d.capitulo_cid
ORDER BY internacoes DESC
LIMIT 10
""").show(truncate=False)

# Top 10 códigos CID-10 por volume, com o valor médio da AIH.
spark.sql(f"""
SELECT f.sk_diagnostico AS codigo_cid10,
       COUNT(*) AS internacoes,
       ROUND(AVG(f.VAL_TOT), 2) AS valor_medio
FROM {base}.fato_internacoes f
GROUP BY f.sk_diagnostico
ORDER BY internacoes DESC
LIMIT 10
""").show(truncate=False)

### Pergunta 4: Existe sazonalidade nas internações ao longo do ano?

In [0]:
# Distribuição mensal das internações, usando a dimensão tempo.
spark.sql(f"""
SELECT t.mes, t.nome_mes AS mes_nome,
       COUNT(*) AS internacoes
FROM {base}.fato_internacoes f
JOIN {base}.dim_tempo t ON f.sk_tempo = t.sk_tempo
GROUP BY t.mes, t.nome_mes
ORDER BY t.mes
""").show(truncate=False)

# Distribuição por dia da semana, com a permanência média de cada dia.
spark.sql(f"""
SELECT t.dia_semana,
       COUNT(*) AS internacoes,
       ROUND(AVG(f.permanencia_dias), 2) AS permanencia_media
FROM {base}.fato_internacoes f
JOIN {base}.dim_tempo t ON f.sk_tempo = t.sk_tempo
GROUP BY t.dia_semana
ORDER BY internacoes DESC
""").show(truncate=False)

### Pergunta 5: Qual a permanência média e como ela varia por diagnóstico?

In [0]:
# Estatísticas gerais de permanência em dias.
spark.sql(f"""
SELECT ROUND(AVG(permanencia_dias), 2) AS permanencia_media,
       ROUND(PERCENTILE(permanencia_dias, 0.5), 0) AS mediana,
       MAX(permanencia_dias) AS permanencia_maxima
FROM {base}.fato_internacoes
""").show(truncate=False)

# Permanência média dos diagnósticos mais frequentes.
# O filtro COUNT(*) >= 100 evita que diagnósticos raros, com poucos
# registros e muita variação, distorçam a comparação.
spark.sql(f"""
SELECT f.sk_diagnostico AS codigo_cid10,
       COUNT(*) AS internacoes,
       ROUND(AVG(f.permanencia_dias), 2) AS permanencia_media
FROM {base}.fato_internacoes f
GROUP BY f.sk_diagnostico
HAVING COUNT(*) >= 100
ORDER BY permanencia_media DESC
LIMIT 10
""").show(truncate=False)

### Pergunta 6: Quais os custos totais e médios das internações?

In [0]:
# Totais gerais de valor das internações.
spark.sql(f"""
SELECT COUNT(*) AS internacoes,
       ROUND(SUM(VAL_TOT), 2)  AS valor_total,
       ROUND(AVG(VAL_TOT), 2)  AS valor_medio
FROM {base}.fato_internacoes
""").show(truncate=False)

# Evolução mensal de custo, cruzando com a dimensão tempo.
spark.sql(f"""
SELECT t.mes, t.nome_mes AS mes_nome,
       ROUND(SUM(f.VAL_TOT), 2) AS valor_total,
       ROUND(AVG(f.VAL_TOT), 2) AS valor_medio
FROM {base}.fato_internacoes f
JOIN {base}.dim_tempo t ON f.sk_tempo = t.sk_tempo
GROUP BY t.mes, t.nome_mes
ORDER BY t.mes
""").show(truncate=False)

# Top 10 municípios de residência por valor total.
spark.sql(f"""
SELECT m.codigo_ibge,
       COUNT(*) AS internacoes,
       ROUND(SUM(f.VAL_TOT), 2) AS valor_total
FROM {base}.fato_internacoes f
JOIN {base}.dim_municipio m ON f.sk_municipio = m.sk_municipio
GROUP BY m.codigo_ibge
ORDER BY valor_total DESC
LIMIT 10
""").show(truncate=False)

### Complemento: taxa de mortalidade e uso de UTI

In [0]:
# Taxa geral de mortalidade nas internações.
spark.sql(f"""
SELECT COUNT(*) AS internacoes,
       SUM(indicador_obito) AS obitos,
       ROUND(100.0 * SUM(indicador_obito) / COUNT(*), 2) AS taxa_mortalidade_pct
FROM {base}.fato_internacoes
""").show(truncate=False)

# Taxa de mortalidade por diagnóstico, limitada aos mais frequentes.
spark.sql(f"""
SELECT f.sk_diagnostico AS codigo_cid10,
       COUNT(*) AS internacoes,
       SUM(f.indicador_obito) AS obitos,
       ROUND(100.0 * SUM(f.indicador_obito) / COUNT(*), 2) AS taxa_mortalidade_pct
FROM {base}.fato_internacoes f
GROUP BY f.sk_diagnostico
HAVING COUNT(*) >= 100
ORDER BY taxa_mortalidade_pct DESC
LIMIT 10
""").show(truncate=False)

# Uso de UTI e comparação de custo médio entre internações com e sem UTI.
spark.sql(f"""
SELECT COUNT(*) AS internacoes,
       SUM(indicador_uti) AS com_uti,
       ROUND(100.0 * SUM(indicador_uti) / COUNT(*), 2) AS pct_uti,
       ROUND(AVG(CASE WHEN indicador_uti = 1 THEN VAL_TOT END), 2) AS valor_medio_com_uti,
       ROUND(AVG(CASE WHEN indicador_uti = 0 THEN VAL_TOT END), 2) AS valor_medio_sem_uti
FROM {base}.fato_internacoes
""").show(truncate=False)

# Discussão Geral

Conectando as respostas das perguntas ao problema original:

1. Perfil por sexo: comparar a proporção entre os sexos e discutir o
   que isso indica sobre o perfil de uso do SUS em Minas Gerais.
2. Faixa etária: identificar os grupos que concentram as internações e
   relacionar com o perfil esperado de morbidade hospitalar (idosos,
   crianças pequenas).
3. Diagnósticos: os capítulos e códigos CID-10 mais frequentes indicam
   as principais causas de internação, base para ações de saúde pública.
4. Sazonalidade: meses e dias da semana de maior volume apontam
   padrões sazonais (período de inverno, doenças respiratórias).
5. Permanência: a permanência média geral e por diagnóstico permite
   dimensionar ocupação de leitos e custos.
6. Custos: o valor total e médio, por mês e por município, mostra onde
   o recurso público está concentrado.
7. Mortalidade e UTI: a taxa geral e por diagnóstico, junto com o custo
   médio com e sem UTI, fecha o quadro de gravidade e complexidade.

Cada resposta deve ser discutida com os valores obtidos nos outputs das
células A4 a A10 e essas discussões transcritas na seção
"Análise de Dados" do README.